# FraudTrace AI - 04: Cross-Source Forensic Entity Resolution

> **DISCLAIMER: SYNTHETIC DATA ONLY**
> All data used in this notebook is synthetically generated for research, testing, and demonstration purposes. No real police records, personal identification, or banking transactions are included. This system does not make legal guilt determinations; it reconstructs and correlates evidentiary relationships.

### Purpose
Resolve disparate entity representations (phones, UPIs, account numbers, names) into unified forensic entities.

| Attribute | Description |
| :--- | :--- |
| **Input** | Raw entity records across multiple sources |
| **Output** | Resolved entity clusters with match confidence scores |
| **Target Environment** | Google Colab (CPU / GPU compatible) & Local |
| **Package** | `ml.src` |


In [ ]:
# ====================================================================
# GOOGLE COLAB ENVIRONMENT SETUP
# ====================================================================
import os
import sys

# Clone repository if running inside Google Colab
if 'google.colab' in str(get_ipython()):
    if not os.path.exists('FraudTrace.AI'):
        !git clone https://github.com/Cyberdude441/FraudTrace.AI.git
    %cd FraudTrace.AI
    !pip install -r ml/requirements-colab.txt

# Ensure repository root is on sys.path for local and Colab execution
repo_root = os.path.abspath('.')
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print(f"[OK] Environment initialized. Working directory: {os.getcwd()}")


## 1. Entity Resolver Architecture
The entity resolver operates on canonical rules and similarity metrics:
- Exact canonical match (e.g. normalized phone numbers)
- Prefix/Suffix account containment
- UPI VPA case-insensitive matching
- Levenshtein string similarity for person names

In [ ]:
from ml.src.resolution.entity_resolution import EntityResolver

resolver = EntityResolver()

test_cases = [
    ("Phone Match (Country code vs local)",
     {"type": "phone", "value": "+91 98765-43210"},
     {"type": "phone", "value": "09876543210"}),
    ("UPI Match (Case & formatting)",
     {"type": "upi", "value": "Merchant.Pay@okhdfcbank"},
     {"type": "upi", "value": "merchant.pay@okhdfcbank"}),
    ("Bank Account Containment",
     {"type": "account_number", "value": "HDFC0001234-987654321"},
     {"type": "account_number", "value": "987654321"}),
    ("Name Fuzzy Match",
     {"type": "name", "value": "Rajesh Kumar Sharma"},
     {"type": "name", "value": "Rajesh K Sharma"}),
    ("Distinct Entities (Should not match)",
     {"type": "phone", "value": "9876543210"},
     {"type": "phone", "value": "9123456789"})
]

print("=== Entity Resolution Test Suite ===")
for label, e1, e2 in test_cases:
    result = resolver.resolve(e1, e2)
    print(f"\nCase: {label}")
    print(f" - E1: {e1['value']} | E2: {e2['value']}")
    print(f" - Same Entity : {result['same_entity']}")
    print(f" - Confidence  : {result['confidence']:.2f}")
    print(f" - Features    : {result['matching_features']}")


## 2. Evaluation Against Master Matches Ground Truth
We test resolver performance against `ml/data/sample/master/matches.csv`.

In [ ]:
from ml.src.data.loader import DataLoader
import pandas as pd

loader = DataLoader(dataset_type="sample")
matches_df = loader.load_master_tables().get('matches', pd.DataFrame())
entities_df = loader.load_master_tables().get('entities', pd.DataFrame())

print(f"Total Master Entity Pairs to Evaluate: {len(matches_df)}")
if not matches_df.empty and not entities_df.empty:
    ent_dict = entities_df.set_index('entity_id').to_dict('index')
    correct = 0
    total = 0
    for _, row in matches_df.iterrows():
        id1, id2 = row['entity_id_1'], row['entity_id_2']
        expected = bool(row['is_match'])
        if id1 in ent_dict and id2 in ent_dict:
            e1 = {'type': ent_dict[id1].get('entity_type', 'unknown'), 'value': ent_dict[id1].get('normalized_value', '')}
            e2 = {'type': ent_dict[id2].get('entity_type', 'unknown'), 'value': ent_dict[id2].get('normalized_value', '')}
            pred = resolver.resolve(e1, e2)['same_entity']
            if pred == expected:
                correct += 1
            total += 1
    if total > 0:
        accuracy = correct / total
        print(f"Entity Resolution Accuracy on Master Benchmark: {accuracy * 100:.1f}% ({correct}/{total})")
